In [1]:
!pip install h5py

In [2]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
from pathlib import Path
import torch.nn as nn
from tqdm.notebook import tqdm 
import torchvision.models as models

In [3]:
class Datas(Dataset):
    def __init__(self,idir,ddir):
        self.idirs=Path(idir)
        self.ddirs=Path(ddir)
        self.idir=sorted(list(self.idirs.glob('*.npy')))
        self.ddir=sorted(list(self.ddirs.glob('*.npy')))
    def __len__(self):
        return len(self.idir)
    def __getitem__(self,idx):
        ip=self.idir[idx]
        dp=self.ddir[idx]
        inp=np.load(ip,allow_pickle=True)
        dnp=np.load(dp,allow_pickle=True)
        if inp.max()>1.0:
            inp=inp/255.0
        it=torch.from_numpy(inp).float()
        dt=torch.from_numpy(dnp).float()
        it=it.permute(2,0,1)
        dt=dt.unsqueeze(0)
        return it,dt

In [4]:
data=Datas('/kaggle/input/nyuv2-labled-dataset/images','/kaggle/input/nyuv2-labled-dataset/depths')

In [5]:
load=DataLoader(data,batch_size=4,shuffle=True)

In [7]:
class DepthNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc=models.resnet18(pretrained=True)
        self.base=list(self.enc.children())[:-2]
        self.encb=nn.Sequential(*self.base)

        self.dec=nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(32, 1, kernel_size=1),
            nn.Sigmoid()
        )
    def forward(self,x):
        feat=self.encb(x)
        dm=self.dec(feat)
        dm = nn.functional.interpolate(dm, size=x.shape[2:], mode='bilinear', align_corners=True)
        return dm*10.0

In [8]:
def lossfn(x,y):
    mask=y>0
    y=y[mask]
    x=x[mask]
    x=torch.clamp(x,min=1e-3)
    y=torch.clamp(y,min=1e-3)
    d=torch.log(x)-torch.log(y)
    t1=torch.mean(d**2)
    t2=(torch.mean(d))**2
    loss=t1-0.5*t2
    return loss

In [9]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [10]:
model=DepthNet().to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 175MB/s] 


In [11]:
optim=torch.optim.Adam(model.parameters(),lr=1e-4)
epochs=7

In [13]:
for epoch in range(epochs):
    model.train()
    los=0.0
    loop=tqdm(load,desc=f'{epoch+1}/{epochs}')
    for imgs,dpts in loop:
        imgs=imgs.to(device)
        dpts=dpts.to(device)
        pred=model(imgs)
        loss=lossfn(pred,dpts)
        optim.zero_grad()
        loss.backward()
        optim.step()
        los+=loss.item()
        loop.set_postfix(loss=loss.item())
    print(f"Epoch {epoch+1} Avg Loss: {los/len(load):.4f}")

1/7:   0%|          | 0/363 [00:00<?, ?it/s]

Epoch 1 Avg Loss: 0.0760


2/7:   0%|          | 0/363 [00:00<?, ?it/s]

Epoch 2 Avg Loss: 0.0556


3/7:   0%|          | 0/363 [00:00<?, ?it/s]

...

Epoch 5 Avg Loss: 0.0275


6/7:   0%|          | 0/363 [00:00<?, ?it/s]

Epoch 6 Avg Loss: 0.0249


7/7:   0%|          | 0/363 [00:00<?, ?it/s]

Epoch 7 Avg Loss: 0.0215


In [14]:
def error(x,y):
    mask=(y>0)&(y<10)
    y=y[mask]
    x=x[mask]
    thresh=torch.max((y/x),(x/y))
    a1=(thresh<1.25).float().mean()
    a2=(thresh<1.25**2).float().mean()
    a3=(thresh<1.25**3).float().mean()
    rmse=(y-x)**2
    rmse=torch.sqrt(rmse.mean())
    lrmse=(torch.log(y)-torch.log(x))**2
    lrmse=torch.sqrt(lrmse.mean())
    arel=torch.mean(torch.abs(y-x)/y)
    sqrel=torch.mean(((y-x)**2)/y)
    return arel,sqrel,rmse,lrmse,a1,a2,a3

In [30]:
def evalu(model,load,device):
    model.eval()
    tabs_rel = 0
    tsq_rel = 0
    trmse = 0
    trmse_log = 0
    ta1 = 0
    ta2 = 0
    ta3 = 0
    count = 0
    with torch.no_grad(): # No gradients needed for eval (saves RAM)
        for images, depths in tqdm(load, desc="Evaluating"):
            images = images.to(device)
            depths = depths.to(device)
            
            # Predict
            preds = model(images)
            
            # Calculate metrics for this batch
            abs_rel, sq_rel, rmse, rmse_log, a1, a2, a3 = error(preds, depths)
            
            # Accumulate (use .item() to get python float)
            tabs_rel += abs_rel.item()
            tsq_rel += sq_rel.item()
            trmse += rmse.item()
            trmse_log += rmse_log.item()
            ta1 += a1.item()
            ta2 += a2.item()
            ta3 += a3.item()
            
            count += 1
    print("\n--- Final Evaluation Metrics ---")
    print(f"Abs Rel:  {tabs_rel / count:.4f}  (Lower is better)")
    print(f"Sq Rel:   {tsq_rel / count:.4f}   (Lower is better)")
    print(f"RMSE:     {trmse / count:.4f}     (Lower is better)")
    print(f"RMSE log: {trmse_log / count:.4f} (Lower is better)")
    print("-" * 30)
    print(f"a1 (δ < 1.25):    {ta1 / count:.4f} (Higher is better)")
    print(f"a2 (δ < 1.25^2):  {ta2 / count:.4f} (Higher is better)")
    print(f"a3 (δ < 1.25^3):  {ta3 / count:.4f} (Higher is better)")